In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 01_make_splits — Direct-Horizon Train/Test Splits (CHN h=2, USA h=3)

**Purpose.** Create leak-free, *direct-horizon* splits for all four segments:
- CHN Export (h=2), CHN Import (h=2)
- USA Export (h=3), USA Import (h=3)

**Design principles**
- Target is `y_target = y(t + h)` within each series (origin, destination, hs6, trade_flow).
- **China**: available data up to **Aug 2025** → test context @ **Aug 2025** → predict **Oct 2025**.
- **USA**: available data up to **Jul 2025** → test context @ **Jul 2025** → predict **Oct 2025**.
- Train rows = months `t` where `y(t+h)` exists in data (non-null).
- No lookahead: all features were built using data ≤ t−1 (enforced in feature notebook).
- No pruning of HS6 series .

**Outputs (all suffixed `_final` in `/data/features/`):**
- `features_CHN_export_train_h2_final.parquet`
- `features_CHN_export_test_h2_final.parquet`
- `features_CHN_import_train_h2_final.parquet`
- `features_CHN_import_test_h2_final.parquet`
- `features_USA_export_train_h3_final.parquet`
- `features_USA_export_test_h3_final.parquet`
- `features_USA_import_train_h3_final.parquet`
- `features_USA_import_test_h3_final.parquet`

In [2]:
# Core imports
import os, json, hashlib, textwrap, random
from datetime import datetime
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

# === Paths (adjust BASE_DIR if needed) ===
BASE_DIR = "/content/drive/MyDrive/ai4trade"  # change if your mount differs
DATA_FEATURES_DIR = f"{BASE_DIR}/data/features"
LOGS_DIR = f"{BASE_DIR}/logs"
RUNS_DIR = f"{LOGS_DIR}/runs"

os.makedirs(DATA_FEATURES_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# === Segment feature inputs (produced by 00_feature_engineering.ipynb) ===
# If your filenames differ, update here:
FEATURE_FILES = {
    "CHN_export": f"{DATA_FEATURES_DIR}/features_CHN_export.parquet",
    "CHN_import": f"{DATA_FEATURES_DIR}/features_CHN_import.parquet",
    "USA_export": f"{DATA_FEATURES_DIR}/features_USA_export.parquet",
    "USA_import": f"{DATA_FEATURES_DIR}/features_USA_import.parquet",
}

# === Horizon & cutoffs ===
# IMPORTANT: 't_last_context' = latest month with complete features; model predicts y(t_last_context + h)
SEGMENT_CFG = {
    "CHN_export": {"h": 2, "t_last_context": "2025-08-01", "origin": "CHN", "flow": "Export"},
    "CHN_import": {"h": 2, "t_last_context": "2025-08-01", "origin": "CHN", "flow": "Import"},
    "USA_export": {"h": 3, "t_last_context": "2025-07-01", "origin": "USA", "flow": "Export"},
    "USA_import": {"h": 3, "t_last_context": "2025-07-01", "origin": "USA", "flow": "Import"},
}

# Derived: max_train_t = t_last_context - h months (so that y(t+h) exists)
def minus_months(ts: pd.Timestamp, m: int) -> pd.Timestamp:
    return (ts.to_period("M") - m).to_timestamp()

for seg, cfg in SEGMENT_CFG.items():
    t_last = pd.to_datetime(cfg["t_last_context"])
    cfg["max_train_t"] = minus_months(t_last, cfg["h"])

RUN_ID = datetime.now().strftime("splits_%Y%m%d_%H%M%S")

# Canonical key order and output column ordering
KEY_COLS = ["origin", "destination", "hs6", "hs4", "trade_flow", "month"]
TARGET_COLS = ["y", "y_target", "horizon"]
# all other features will be preserved as-is after these

print("RUN_ID:", RUN_ID)
pd.DataFrame(SEGMENT_CFG).T

RUN_ID: splits_20251030_053628


,h,t_last_context,origin,flow,max_train_t
CHN_export,2,2025-08-01,CHN,Export,2025-06-01 00:00:00
CHN_import,2,2025-08-01,CHN,Import,2025-06-01 00:00:00
USA_export,3,2025-07-01,USA,Export,2025-04-01 00:00:00
USA_import,3,2025-07-01,USA,Import,2025-04-01 00:00:00


## Utilities

We define helpers for:
- reading parquet and normalizing schema/dtypes,
- enforcing unique keys and deduplicating duplicate column names,
- constructing direct-horizon targets (`y_target = y(t+h)`),
- splitting into train/test by explicit calendar rules,
- QA checks (null targets, key uniqueness, feature freshness spot-check),
- saving files and logging a compact JSON run record.

In [3]:
def _normalize_month_col(df: pd.DataFrame, col="month") -> pd.DataFrame:
    # Robust month parsing to first-of-month
    if not np.issubdtype(df[col].dtype, np.datetime64):
        df[col] = pd.to_datetime(df[col], errors="coerce")
    # force to first day of month
    df[col] = df[col].dt.to_period("M").dt.to_timestamp()
    return df

def _dedupe_columns(df: pd.DataFrame) -> pd.DataFrame:
    # If duplicate column labels exist, keep the first occurrence
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated(keep="first")]
    return df

def _enforce_key_uniqueness(df: pd.DataFrame, key_cols=KEY_COLS, strict=True) -> None:
    dups = df.duplicated(subset=key_cols, keep=False)
    if dups.any():
        msg = f"Found {dups.sum()} duplicate rows on keys {key_cols}. Inspect before proceeding."
        if strict:
            raise ValueError(msg)
        else:
            print("WARN:", msg)

def _direct_horizon_target(df: pd.DataFrame, h: int) -> pd.DataFrame:
    # Assumes df contains: origin, destination, hs6, trade_flow, month, y
    # Sort for deterministic shift and groupby
    df = df.sort_values(["origin", "destination", "hs6", "trade_flow", "month"])
    grp = df.groupby(["origin", "destination", "hs6", "trade_flow"], sort=False, group_keys=False)
    df["y_target"] = grp["y"].shift(-h)  # y(t+h)
    return df

def _make_splits(df: pd.DataFrame, h: int, t_last_context: str, max_train_t: str):
    t_last = pd.to_datetime(t_last_context)
    t_train_max = pd.to_datetime(max_train_t)

    # TRAIN: rows with y_target not null AND month ≤ max_train_t
    train_mask = (df["y_target"].notna()) & (df["month"] <= t_train_max)
    train_df = df.loc[train_mask].copy()

    # TEST: rows at t_last_context (live inference context). y_target should be null here by design.
    test_mask = df["month"] == t_last
    test_df = df.loc[test_mask].copy()

    # Add horizon for traceability
    train_df["horizon"] = h
    test_df["horizon"] = h

    return train_df, test_df

def _ordered_columns(df: pd.DataFrame) -> list:
    # Keep keys + targets first, then the rest (features)
    pref = KEY_COLS + TARGET_COLS
    rest = [c for c in df.columns if c not in pref]
    return pref + rest

def _save_parquet(df: pd.DataFrame, path: str) -> dict:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)
    # cheap file hash for provenance
    with open(path, "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()
    return {"path": path, "rows": len(df), "md5": md5}

def _lag_freshness_spotcheck(df: pd.DataFrame, lag_col="lag_1", sample_groups=3, seed=7):
    """
    Spot check: for a few random series ensure lag_1(t) == y(t-1).
    Non-fatal; prints info to help sanity-check feature freshness.
    """
    if lag_col not in df.columns:
        print(f"[SpotCheck] Skipped: '{lag_col}' not present.")
        return
    keys = ["origin", "destination", "hs6", "trade_flow"]
    rng = random.Random(seed)

    # pick a few groups with ≥ 3 rows
    grp_sizes = df.groupby(keys).size()
    candidates = grp_sizes[grp_sizes >= 3].index.tolist()
    rng.shuffle(candidates)
    for key in candidates[:sample_groups]:
        sub = df.loc[(df[keys[0]]==key[0]) & (df[keys[1]]==key[1]) & (df[keys[2]]==key[2]) & (df[keys[3]]==key[3])].sort_values("month")
        ok = True
        for i in range(1, min(6, len(sub))):  # check a few consecutive months
            y_prev = sub["y"].iloc[i-1]
            lag1   = sub["lag_1"].iloc[i]
            if pd.notna(lag1) and pd.notna(y_prev) and not np.isclose(lag1, y_prev):
                ok = False
                break
        print(f"[SpotCheck] {key} → {'OK' if ok else 'MISMATCH'}; rows={len(sub)}")

def _summary_line(df: pd.DataFrame, name: str) -> dict:
    return {
        "name": name,
        "rows": int(len(df)),
        "month_min": str(df["month"].min()) if len(df) else None,
        "month_max": str(df["month"].max()) if len(df) else None,
        "y_target_nulls": int(df["y_target"].isna().sum()) if "y_target" in df.columns else None,
        "unique_keys": int(df.drop_duplicates(KEY_COLS).shape[0]),
    }

## Segment plan (derived)

For each segment we compute:
- `h` (horizon),
- `t_last_context` (latest observed month with complete features),
- `max_train_t = t_last_context - h months` (last month allowed in TRAIN so that `y(t+h)` exists).

This table is our **single source of truth** for split boundaries.

In [4]:
plan_rows = []
for seg, cfg in SEGMENT_CFG.items():
    row = {
        "segment": seg,
        "origin": cfg["origin"],
        "flow": cfg["flow"],
        "h": cfg["h"],
        "t_last_context": cfg["t_last_context"],
        "max_train_t": str(cfg["max_train_t"]),
        "predicts": str((pd.to_datetime(cfg["t_last_context"]).to_period("M") + cfg["h"]).to_timestamp()),
    }
    plan_rows.append(row)
plan_df = pd.DataFrame(plan_rows).sort_values(["origin","flow"])
plan_df

,segment,origin,flow,h,t_last_context,max_train_t,predicts
0,CHN_export,CHN,Export,2,2025-08-01,2025-06-01 00:00:00,2025-10-01 00:00:00
1,CHN_import,CHN,Import,2,2025-08-01,2025-06-01 00:00:00,2025-10-01 00:00:00
2,USA_export,USA,Export,3,2025-07-01,2025-04-01 00:00:00,2025-10-01 00:00:00
3,USA_import,USA,Import,3,2025-07-01,2025-04-01 00:00:00,2025-10-01 00:00:00


## Split creation loop

For each segment:
1. Load features parquet.
2. Normalize schema (month to first-of-month, dedupe columns).
3. Enforce uniqueness of (origin, destination, hs6, trade_flow, month).
4. Build `y_target = y(t+h)`.
5. Split into TRAIN (non-null target, month ≤ max_train_t) and TEST (month == t_last_context).
6. Run QA checks:
   - TRAIN: no null `y_target`, month range ≤ `max_train_t`.
   - TEST: all rows at `t_last_context`; `y_target` should be null.
   - Key uniqueness for both.
   - Spot-check `lag_1` freshness on TRAIN.
7. Save to parquet with `_final` suffix.
8. Record run metadata for a JSON log.

In [5]:
outputs = []
summaries = []

for segment, in_path in FEATURE_FILES.items():
    cfg = SEGMENT_CFG[segment]
    h = cfg["h"]
    t_last_context = cfg["t_last_context"]
    max_train_t = str(cfg["max_train_t"])

    # === 1) Load ===
    print(f"\n=== [{segment}] Loading ===")
    if not os.path.exists(in_path):
        raise FileNotFoundError(f"Missing features file: {in_path}")
    df = pd.read_parquet(in_path)

    # === 2) Normalize schema ===
    df = _dedupe_columns(df)
    if "y" not in df.columns:
        raise KeyError("Expected target column 'y' not found in features file.")
    for col in ["origin","destination","hs6","hs4","trade_flow"]:
        if col not in df.columns:
            raise KeyError(f"Missing required column '{col}' in features file.")
    df = _normalize_month_col(df, "month")

    # === 3) Unique key guard ===
    _enforce_key_uniqueness(df, KEY_COLS, strict=True)

    # === 4) Build direct-horizon target ===
    df = _direct_horizon_target(df, h=h)

    # === 5) Make splits ===
    train_df, test_df = _make_splits(df, h=h, t_last_context=t_last_context, max_train_t=max_train_t)

    # Final column ordering (keys + targets first)
    train_df = train_df[_ordered_columns(train_df)]
    test_df  = test_df[_ordered_columns(test_df)]

    # === 6) QA checks ===
    # TRAIN: no null y_target
    nulls_train = train_df["y_target"].isna().sum()
    assert nulls_train == 0, f"[{segment}] TRAIN has {nulls_train} null y_target values — investigate."
    # TRAIN: month ≤ max_train_t
    assert train_df["month"].max() <= pd.to_datetime(max_train_t), f"[{segment}] TRAIN month exceeds {max_train_t}"
    # TEST: all rows at t_last_context
    if len(test_df):
        assert (test_df["month"] == pd.to_datetime(t_last_context)).all(), f"[{segment}] TEST contains months != {t_last_context}"
        # TEST: y_target should be null (not strictly required but expected)
        test_nulls = test_df["y_target"].isna().sum()
        if test_nulls != len(test_df):
            print(f"[WARN][{segment}] TEST has {len(test_df)-test_nulls} non-null y_target rows (ok if true target exists).")
    # Uniqueness
    _enforce_key_uniqueness(train_df, KEY_COLS, strict=True)
    _enforce_key_uniqueness(test_df, KEY_COLS, strict=True)

    # Spot-check lag_1 freshness on TRAIN (non-fatal)
    _lag_freshness_spotcheck(train_df, lag_col="lag_1", sample_groups=3, seed=7)

    # === 7) Save outputs ===
    out_train = f"{DATA_FEATURES_DIR}/features_{segment.split('_')[0]}_{segment.split('_')[1]}_train_h{h}_final.parquet"
    out_test  = f"{DATA_FEATURES_DIR}/features_{segment.split('_')[0]}_{segment.split('_')[1]}_test_h{h}_final.parquet"

    meta_train = _save_parquet(train_df, out_train)
    meta_test  = _save_parquet(test_df,  out_test)

    outputs.extend([meta_train, meta_test])

    # === 8) Summaries for on-screen + log ===
    summaries.append(_summary_line(train_df, name=os.path.basename(out_train)))
    summaries.append(_summary_line(test_df,  name=os.path.basename(out_test)))

print("\n✔ Done splitting all segments.")
pd.DataFrame(summaries)


=== [CHN_export] Loading ===
[SpotCheck] ('CHN', 'AUS', '610899', 'Export') → OK; rows=28
[SpotCheck] ('CHN', 'SGP', '640192', 'Export') → OK; rows=30
[SpotCheck] ('CHN', 'TUR', '851830', 'Export') → OK; rows=30

=== [CHN_import] Loading ===
[SpotCheck] ('CHN', 'SGP', '330112', 'Import') → OK; rows=17
[SpotCheck] ('CHN', 'ZAF', '853310', 'Import') → OK; rows=4
[SpotCheck] ('CHN', 'IND', '901920', 'Import') → OK; rows=20

=== [USA_export] Loading ===
[SpotCheck] ('USA', 'THA', '030616', 'Export') → OK; rows=23
[SpotCheck] ('USA', 'CHL', '681519', 'Export') → OK; rows=28
[SpotCheck] ('USA', 'GBR', '420292', 'Export') → OK; rows=28

=== [USA_import] Loading ===
[SpotCheck] ('USA', 'ISR', '848130', 'Import') → OK; rows=28
[SpotCheck] ('USA', 'ISR', '691410', 'Import') → OK; rows=21
[SpotCheck] ('USA', 'MEX', '890311', 'Import') → OK; rows=28

✔ Done splitting all segments.


,name,rows,month_min,month_max,y_target_nulls,unique_keys
0,features_CHN_export_train_h2_final.parquet,2807460,2023-01-01 00:00:00,2025-06-01 00:00:00,0,2807460
1,features_CHN_export_test_h2_final.parquet,94698,2025-08-01 00:00:00,2025-08-01 00:00:00,94698,94698
2,features_CHN_import_train_h2_final.parquet,1118879,2023-01-01 00:00:00,2025-06-01 00:00:00,0,1118879
3,features_CHN_import_test_h2_final.parquet,35732,2025-08-01 00:00:00,2025-08-01 00:00:00,35732,35732
4,features_USA_export_train_h3_final.parquet,2257793,2023-01-01 00:00:00,2025-04-01 00:00:00,0,2257793
5,features_USA_export_test_h3_final.parquet,61550,2025-07-01 00:00:00,2025-07-01 00:00:00,61550,61550
6,features_USA_import_train_h3_final.parquet,1952673,2023-01-01 00:00:00,2025-04-01 00:00:00,0,1952673
7,features_USA_import_test_h3_final.parquet,57393,2025-07-01 00:00:00,2025-07-01 00:00:00,57393,57393


In [6]:
# Inspect month coverage per segment + check if the context month is present at all
def month_coverage_report(path, t_last_context):
    if not os.path.exists(path):
        print("MISSING:", path);
        return None
    df = pd.read_parquet(path, columns=["origin","trade_flow","month"])
    # normalize month to first of month
    df["month"] = pd.to_datetime(df["month"]).dt.to_period("M").dt.to_timestamp()
    months = df["month"].drop_duplicates().sort_values().tolist()
    print(os.path.basename(path))
    print("  months:", len(months), "from", months[0] if months else None, "to", months[-1] if months else None)
    present = pd.Timestamp(t_last_context) in set(months)
    print("  has_context_month", t_last_context, "→", present)
    return months

for seg, path in FEATURE_FILES.items():
    t_last = SEGMENT_CFG[seg]["t_last_context"]
    month_coverage_report(path, t_last)

features_CHN_export.parquet
  months: 32 from 2023-01-01 00:00:00 to 2025-08-01 00:00:00
  has_context_month 2025-08-01 → True
features_CHN_import.parquet
  months: 32 from 2023-01-01 00:00:00 to 2025-08-01 00:00:00
  has_context_month 2025-08-01 → True
features_USA_export.parquet
  months: 31 from 2023-01-01 00:00:00 to 2025-07-01 00:00:00
  has_context_month 2025-07-01 → True
features_USA_import.parquet
  months: 31 from 2023-01-01 00:00:00 to 2025-07-01 00:00:00
  has_context_month 2025-07-01 → True


In [7]:
# Check where y_target becomes NaN and how far your TRAIN could go BEFORE we applied our max_train_t cap.
# This tells whether the 'shift(-h)' itself is already running out earlier than it should.

def inspect_shift_ceiling(path, h):
    if not os.path.exists(path):
        print("MISSING:", path);
        return
    df = pd.read_parquet(path, columns=["origin","destination","hs6","trade_flow","month","y"])
    df["month"] = pd.to_datetime(df["month"]).dt.to_period("M").dt.to_timestamp()
    df = df.sort_values(["origin","destination","hs6","trade_flow","month"])
    grp = df.groupby(["origin","destination","hs6","trade_flow"], sort=False, group_keys=False)
    df["y_target_tmp"] = grp["y"].shift(-h)
    # Latest month for which y_target_tmp is NOT NaN
    ceiling = df.loc[df["y_target_tmp"].notna(), "month"].max()
    print(os.path.basename(path), "| h=", h, "| last month with non-null shifted target =", ceiling)

for seg, path in FEATURE_FILES.items():
    h = SEGMENT_CFG[seg]["h"]
    inspect_shift_ceiling(path, h)

features_CHN_export.parquet | h= 2 | last month with non-null shifted target = 2025-06-01 00:00:00
features_CHN_import.parquet | h= 2 | last month with non-null shifted target = 2025-06-01 00:00:00
features_USA_export.parquet | h= 3 | last month with non-null shifted target = 2025-04-01 00:00:00
features_USA_import.parquet | h= 3 | last month with non-null shifted target = 2025-04-01 00:00:00


## Run log (JSON)

We store a compact log capturing:
- run id / timestamp,
- segment configs (h, t_last_context, max_train_t),
- output files with md5 hashes and row counts,
- split summaries (row counts, date ranges),
- code provenance note.

The log is saved to `/logs/runs/{RUN_ID}.json`.

In [9]:
import json
from datetime import datetime
import pandas as pd
import numpy as np

# Convert anything not JSON-serializable (Timestamp, numpy types) to plain Python types
def _json_safe(o):
    if isinstance(o, (pd.Timestamp, np.datetime64)):
        return str(pd.Timestamp(o))
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    return str(o)

run_log = {
    "run_id": RUN_ID,
    "created_at_ist": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "notebook": "01_make_splits.ipynb",
    "policy": {
        "china_h": 2,
        "usa_h": 3,
        "note": "Direct-horizon targets; test = live inference context (Aug-2025 CHN, Jul-2025 USA). No pruning."
    },
    "segments": SEGMENT_CFG,    # contains timestamps
    "outputs": outputs,         # contains md5 hashes and file paths
    "summaries": summaries,     # contains timestamps in string form already
    "provenance": "Columns kept: keys + y + y_target + horizon + all feature columns; strict key uniqueness; lag_1 freshness spot-check."
}

log_path = f"{RUNS_DIR}/{RUN_ID}.json"
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=2, default=_json_safe)

print("Wrote log:", log_path)


Wrote log: /content/drive/MyDrive/ai4trade/logs/runs/splits_20251030_053628.json


## Quick “next steps” (for your future self)

1) Model training (segment-by-segment):
   - `12_xgb_tweedie.ipynb`
   - `13_xgb_log1p.ipynb`
   - `10_lgbm_rmse.ipynb`
   - Save OOF + Forecast to `/predictions/oof` and `/predictions/forecast` with `_final`.

2) Validation:
   - Use horizon folds: CHN (C1–C6), USA (U1–U5) with later-fold upweights.
   - Evaluate sMAPE on **HS4 aggregates** to align with submission layer.

3) Blending:
   - `40_blend_weights.ipynb` → inverse-sMAPE weights (floor=0.10) per segment.

4) Ensembles & Submission:
   - Blend at HS6 → aggregate to HS4.
   - Concatenate 4 segments → `final_forecast_hs4_final.parquet`.
   - `50_make_submission.ipynb` → `submission_final.csv` (OEC format).